# Speaker Verification: ResNet-50 + SpecAugment (Cosine Contrastive Loss, 5s Audio)

## What's New vs Previous ResNet-50 Run
| Setting | Previous | **This Run** |
|:--------|:--------:|:------------:|
| Architecture | ResNet-50 | ResNet-50 |
| Audio Window | 5 seconds | 5 seconds |
| Loss | Cosine Contrastive | Cosine Contrastive |
| **SpecAugment** | ❌ None | ✅ **Freq + Time Masking** |

**SpecAugment** randomly masks horizontal (frequency) and vertical (time) strips of the spectrogram during training.
This forces the model to learn speaker identity from multiple complementary features, reducing overfitting.


In [ ]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT   = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
TRAIN_CSV      = os.path.join(DATASET_ROOT, "train_pairs.csv")

TEST_ROOT      = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"
TEST_AUDIO_DIR = os.path.join(TEST_ROOT, "test-clean")
TEST_CSV       = os.path.join(TEST_ROOT, "test_pairs.csv")

print("Train audio dir:", BASE_AUDIO_DIR)
print("Test  audio dir:", TEST_AUDIO_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

def to_train_abs(rel):
    return os.path.join(BASE_AUDIO_DIR, rel.replace("train-clean-100/", "", 1))
def to_test_abs(rel):
    return os.path.join(TEST_AUDIO_DIR, rel)

train_df["path1_abs"] = train_df["audio_path_1"].apply(to_train_abs)
train_df["path2_abs"] = train_df["audio_path_2"].apply(to_train_abs)
test_df["path1_abs"]  = test_df["audio_path_1"].apply(to_test_abs)
test_df["path2_abs"]  = test_df["audio_path_2"].apply(to_test_abs)

print("Train rows:", len(train_df), "| Label dist:")
print(train_df["label"].value_counts().to_dict())
print("Test rows:", len(test_df))

## 3. Audio Transforms — 5s Window + SpecAugment

**SpecAugment** applies two random masks to the Mel Spectrogram during training:
- `FrequencyMasking(freq_mask_param=15)` → masks up to 15 of 80 frequency bins
- `TimeMasking(time_mask_param=50)` → masks up to 50 of ~500 time steps

This is applied **only during training**. During evaluation, the full spectrogram is used.

In [ ]:
import torch
import numpy as np
import torchaudio.transforms as T

TARGET_SR     = 16000
TARGET_LENGTH = TARGET_SR * 5   # 80000 samples (5 seconds)

def crop_or_pad(audio, is_train=True):
    length = len(audio)
    if length > TARGET_LENGTH:
        if is_train:
            start = np.random.randint(0, length - TARGET_LENGTH)
        else:
            start = (length - TARGET_LENGTH) // 2
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        if is_train:
            audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
        # Testing: dynamic length (no padding)
    return audio

mel_transform   = T.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
amplitude_to_db = T.AmplitudeToDB()

# ── SpecAugment ──────────────────────────────────────────────────────────────
freq_mask = T.FrequencyMasking(freq_mask_param=15)   # mask up to 15 of 80 freq bins
time_mask = T.TimeMasking(time_mask_param=50)         # mask up to 50 of ~500 time steps

def apply_spec_augment(mel_spec):
    """Apply SpecAugment: 2x frequency mask + 2x time mask"""
    mel_spec = freq_mask(mel_spec)
    mel_spec = freq_mask(mel_spec)   # apply twice for stronger augmentation
    mel_spec = time_mask(mel_spec)
    mel_spec = time_mask(mel_spec)   # apply twice
    return mel_spec

print("Transforms ready.")
print("SpecAugment: FreqMask(15) x2  +  TimeMask(50) x2")
print("Target duration: 5 seconds")

## 4. SpeakerPairDataset

During training, SpecAugment is applied to each spectrogram after the Mel transform.

In [ ]:
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db, is_train=True):
        self.df              = dataframe
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db
        self.is_train        = is_train

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio, is_train=self.is_train)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel = self.amplitude_to_db(self.mel_transform(audio))
        
        # Apply SpecAugment ONLY during training
        if self.is_train:
            mel = apply_spec_augment(mel)
        return mel

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel1  = self.load_audio(row["path1_abs"])
        mel2  = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

In [ ]:
train_dataset = SpeakerPairDataset(train_df, mel_transform, amplitude_to_db, is_train=True)
test_dataset  = SpeakerPairDataset(test_df,  mel_transform, amplitude_to_db, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=1,  shuffle=False, num_workers=2, pin_memory=True)

print(f"Train dataset: {len(train_dataset):,} pairs | {len(train_loader):,} batches (BS=64)")
print(f"Test dataset:  {len(test_dataset):,} pairs  | {len(test_loader):,} batches (BS=1, dynamic length)")

## 5. Visualize SpecAugment Effect

Shows the same audio clip before and after SpecAugment to see the masked regions.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first training sample
sample_row = train_df.iloc[0]
audio, sr = sf.read(sample_row["path1_abs"])
if len(audio.shape) > 1: audio = np.mean(audio, axis=1)
audio = crop_or_pad(audio, is_train=False)
audio_tensor = torch.tensor(audio).float().unsqueeze(0)
mel_original = amplitude_to_db(mel_transform(audio_tensor))
mel_augmented = apply_spec_augment(mel_original.clone())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('SpecAugment Visualization', fontsize=13, fontweight='bold')

axes[0].imshow(mel_original.squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original Mel Spectrogram')
axes[0].set_xlabel('Time'); axes[0].set_ylabel('Mel Bin')

axes[1].imshow(mel_augmented.squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('After SpecAugment (masked regions)')
axes[1].set_xlabel('Time'); axes[1].set_ylabel('Mel Bin')

plt.tight_layout()
plt.savefig('specaugment_visualization.png', dpi=150)
plt.show()
print('Saved → specaugment_visualization.png')

## 6. Model Architecture — ResNet-50

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 7. Cosine Contrastive Loss

In [ ]:
class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        similarity = F.cosine_similarity(emb1, emb2)
        pos_loss = label * torch.pow(1.0 - similarity, 2)
        neg_loss = (1 - label) * torch.pow(torch.clamp(similarity - self.margin, min=0.0), 2)
        return torch.mean(pos_loss + neg_loss)

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = ResNetSpeaker(embedding_dim=256).to(device)
criterion = CosineContrastiveLoss(margin=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)
print("Model:  ResNet-50 + SpecAugment")
print("Loss:   CosineContrastiveLoss (margin=0.5)")
print("Optim:  Adam (lr=1e-3)")

## 8. Training Loop — 25 Epochs

Auto-saves checkpoint after **every epoch** to protect against Kaggle 12-hour timeout.

In [ ]:
from tqdm import tqdm

NUM_EPOCHS  = 25
PRINT_EVERY = 50

loss_history      = []
train_acc_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss    = 0
    total_correct = 0
    total_samples = 0

    bar = tqdm(enumerate(train_loader), total=len(train_loader),
               desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch_idx, (mel1, mel2, labels) in bar:
        mel1   = mel1.to(device)
        mel2   = mel2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        emb1 = model(mel1)
        emb2 = model(mel2)

        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            sim   = F.cosine_similarity(emb1, emb2)
            preds = (sim > 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

        if batch_idx % PRINT_EVERY == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    avg_acc  = total_correct / total_samples

    loss_history.append(avg_loss)
    train_acc_history.append(avg_acc)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] — Avg Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}\n")

    # ── Auto-save every epoch ──
    torch.save({
        "model_state":        model.state_dict(),
        "epoch":              epoch,
        "optimizer_state":    optimizer.state_dict(),
        "loss_history":       loss_history,
        "train_acc_history":  train_acc_history,
    }, "checkpoint_resnet50_specaugment_5s.pth")
    print(f"  Checkpoint saved → epoch {epoch+1}")

print("\nTraining complete!")

In [ ]:
torch.save({
    "model_state":       model.state_dict(),
    "epoch":             NUM_EPOCHS - 1,
    "optimizer_state":   optimizer.state_dict(),
    "loss_history":      loss_history,
    "train_acc_history": train_acc_history,
}, "checkpoint_resnet50_specaugment_5s.pth")

print("Final model saved → checkpoint_resnet50_specaugment_5s.pth")

## 9. Training History Graphs

In [ ]:
epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ResNet50 + SpecAugment + 5s — Training Metrics", fontsize=13, fontweight='bold')

axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0].set_title("Cosine Contrastive Loss per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, train_acc_history, marker='o', color='steelblue', linewidth=2)
axes[1].set_title("Training Accuracy per Epoch")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics_specaugment_5s.png", dpi=150)
plt.show()
print("Saved → training_metrics_specaugment_5s.png")

## 10. Evaluate on Both Datasets

Uses **cosine similarity > 0.5** to predict Same/Different.

In [ ]:
def evaluate(model, loader, device, label_name="Set"):
    model.eval()
    correct = 0; total = 0
    same_sims, diff_sims = [], []
    same_dists, diff_dists = [], []

    with torch.no_grad():
        for mel1, mel2, labels in tqdm(loader, desc=f"Evaluating {label_name}"):
            mel1   = mel1.to(device)
            mel2   = mel2.to(device)
            labels = labels.to(device)

            emb1 = model(mel1)
            emb2 = model(mel2)

            sim   = F.cosine_similarity(emb1, emb2)
            preds = (sim > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            for s, lbl in zip(sim.cpu().tolist(), labels.cpu().tolist()):
                (same_sims if lbl == 1 else diff_sims).append(s)

            d = F.pairwise_distance(emb1, emb2).cpu().tolist()
            for dist, lbl in zip(d, labels.cpu().tolist()):
                (same_dists if lbl == 1 else diff_dists).append(dist)

    acc = correct / total
    print(f"{label_name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    return acc, same_sims, diff_sims, same_dists, diff_dists

# Need a train loader without SpecAugment for fair evaluation
train_eval_dataset = SpeakerPairDataset(train_df, mel_transform, amplitude_to_db, is_train=False)
train_eval_loader  = DataLoader(train_eval_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

train_acc, tr_same_sim, tr_diff_sim, tr_same_d, tr_diff_d = evaluate(
    model, train_eval_loader, device, "Training Set"
)
test_acc,  te_same_sim, te_diff_sim, te_same_d, te_diff_d = evaluate(
    model, test_loader,  device, "Test Set"
)

print(f"\nTrain Acc: {train_acc*100:.2f}%  |  Test Acc: {test_acc*100:.2f}%")
print(f"Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")

## 11. Accuracy Comparison Bar Chart

In [ ]:
plt.figure(figsize=(7, 5))
bars  = plt.bar(["Training Accuracy", "Test Accuracy"],
                [train_acc, test_acc],
                color=["steelblue", "darkorange"], width=0.4, edgecolor='white')
for bar, val in zip(bars, [train_acc, test_acc]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val*100:.2f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("ResNet50 + SpecAugment 5s — Training vs Test Accuracy", fontsize=12, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_comparison_specaugment_5s.png", dpi=150)
plt.show()
print("Saved → accuracy_comparison_specaugment_5s.png")

## 12. Cosine Similarity Distribution — Training vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Cosine Similarity Distribution (Train vs Test)", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], tr_same_sim[:5000], tr_diff_sim[:5000], "Training Set"),
    (axes[1], te_same_sim[:5000], te_diff_sim[:5000], "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
    ax.set_title(title)
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Count")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cosine_sim_distribution_specaugment_5s.png", dpi=150)
plt.show()
print("Saved → cosine_sim_distribution_specaugment_5s.png")

## 13. Euclidean Distance Distribution — Training vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Euclidean Distance Distribution (Train vs Test)", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], tr_same_d[:5000], tr_diff_d[:5000], "Training Set"),
    (axes[1], te_same_d[:5000], te_diff_d[:5000], "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.set_title(title)
    ax.set_xlabel("Euclidean Distance")
    ax.set_ylabel("Count")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("euclidean_dist_specaugment_5s.png", dpi=150)
plt.show()
print("Saved → euclidean_dist_specaugment_5s.png")

## 14. Final Summary

In [ ]:
print("=" * 50)
print("  MODEL:  ResNet-50 + SpecAugment + Cosine Loss")
print("  AUDIO:  5-second clips (80000 samples)")
print("  AUGMENT: SpecAugment (FreqMask=15x2, TimeMask=50x2)")
print("-" * 50)
print(f"  Training Accuracy : {train_acc*100:.2f}%")
print(f"  Test Accuracy     : {test_acc*100:.2f}%")
print(f"  Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")
print("=" * 50)
print("Saved files:")
print("  checkpoint_resnet50_specaugment_5s.pth")
print("  specaugment_visualization.png")
print("  training_metrics_specaugment_5s.png")
print("  accuracy_comparison_specaugment_5s.png")
print("  cosine_sim_distribution_specaugment_5s.png")
print("  euclidean_dist_specaugment_5s.png")